# cross-product-normal — faded example 2: Triangle areas of a batch via cross-product magnitude

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-product-normal`. Running the beacon reports progress on the `Geometry: Cross-product surface normal` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Cross-product surface normal` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-product-normal`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-product-normal"
DD_SUBTOPIC = "Geometry: Cross-product surface normal"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The magnitude of the cross product of two triangle edges equals twice the triangle's area: `area = 0.5 * |cross(P2-P1, P3-P1)|`. This works in any orientation because the cross-product length is the area of the parallelogram the edges span. Batching over `(N, 3, 3)` triangles avoids a Python loop.

## Faded exercise 2

Implement `batched_areas(tris)` where `tris` is `(N, 3, 3)`. Compute the area of each triangle from the cross-product magnitude of its two edges, returning an `(N,)` tensor. Complete the missing line that turns each edge cross product into a scalar area.

**Fill in:** Compute per-triangle area as half the L2 norm of the cross product along the last axis.

In [ ]:
def batched_areas(tris: Tensor) -> Tensor:
    e1 = tris[:, 1] - tris[:, 0]                    # (N, 3)
    e2 = tris[:, 2] - tris[:, 0]                    # (N, 3)
    n = t.linalg.cross(e1, e2, dim=-1)              # (N, 3)
    areas = None  # TODO: Compute per-triangle area as half the L2 norm of the cross product along the last axis.
    return areas


def _test():
    tris = t.tensor([
        [[0.0, 0.0, 0.0], [2.0, 0.0, 0.0], [0.0, 2.0, 0.0]],   # area 2
        [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]],   # area 0.5
        [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [3.0, 0.0, 0.0]],   # degenerate, area 0
    ])
    out = batched_areas(tris)
    e1 = tris[:, 1] - tris[:, 0]
    e2 = tris[:, 2] - tris[:, 0]
    expected = 0.5 * t.linalg.cross(e1, e2, dim=-1).norm(dim=-1)
    assert out.shape == (3,)
    assert t.allclose(out, expected, atol=1e-6)
    assert t.allclose(out, t.tensor([2.0, 0.5, 0.0]), atol=1e-6)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def batched_areas(tris: Tensor) -> Tensor:
    e1 = tris[:, 1] - tris[:, 0]                    # (N, 3)
    e2 = tris[:, 2] - tris[:, 0]                    # (N, 3)
    n = t.linalg.cross(e1, e2, dim=-1)              # (N, 3)
    areas = 0.5 * n.norm(dim=-1)                    # (N,)
    return areas
```
</details>